# DENTEX 4-Class Abnormal Tooth Detection Benchmark (Google Colab)

This notebook provides a complete 1-click execution pipeline on **Google Colab**:
1. Google Drive mount for checkpoint and artifact persistence
2. Automated environment setup and dependency verification
3. DENTEX data download and COCO 4-class annotation generation
4. Unified benchmark training across 3 architectures (Faster R-CNN, DINO, HierarchicalDet)
5. Evaluation and inline Markdown benchmark report rendering


## 1. Mount Google Drive and Set Working Directory


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount('/content/drive')

# Set working directory on Google Drive
PROJECT_DIR = Path('/content/drive/MyDrive/Dentex')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
%cd {PROJECT_DIR}
sys.path.insert(0, str(PROJECT_DIR))
print(f'Working directory: {os.getcwd()}')

if not (PROJECT_DIR / 'dentex_bench').exists():
    print('Project not found at target directory, cloning repository...')
    !git clone https://github.com/ibrahimethemhamamci/DENTEX.git {PROJECT_DIR}


## 2. Environment Setup & Dependency Verification


In [ ]:
# Install project dependencies and check environment
!pip install -r requirements.txt
!python scripts/setup_env.py


## 3. Dataset Download & COCO 4-Class Preparation


In [ ]:
# Downloads DENTEX dataset from HuggingFace and builds COCO train/val splits
!python scripts/download_and_prepare.py --seed 42


## 4. Benchmark Model Training

Train all three architectures under identical 80/20 train/val splits:
- **Faster R-CNN**: Two-stage CNN baseline (ResNet50-FPN)
- **DINO**: DETR-based Vision Transformer with contrastive denoising
- **HierarchicalDet**: Denoising diffusion object detector baseline


In [ ]:
# Train Faster R-CNN baseline
!python scripts/train.py --model faster_rcnn --epochs 30 --lr 0.005


In [ ]:
# Train DINO Transformer detector
!python scripts/train.py --model dino --epochs 30 --lr 0.0001


In [ ]:
# Train HierarchicalDet Diffusion detector
!python scripts/train.py --model hierarchical_det --epochs 30 --lr 0.0001


## 5. Comprehensive Benchmark Evaluation


In [ ]:
# Evaluate all models and generate comparative benchmark summary
!python scripts/evaluate.py --compare_all


## 6. Render Benchmark Summary Report


In [ ]:
from IPython.display import display, Markdown
from pathlib import Path

summary_path = Path('outputs/benchmark_summary.md')
if summary_path.exists():
    display(Markdown(summary_path.read_text(encoding='utf-8')))
else:
    print('outputs/benchmark_summary.md not found. Ensure evaluation has completed.')
